# 4. Implementation: Transferable UnitModel Architecture

This notebook implements the complete **Domain-Invariant Metric Network (`UnitMetricNet`)**.

### Architecture Highlights:
1. **Latent Hypersphere Embedding Backbone:** Multi-layer deep feature projector with LayerNorm, ReLU, and L2 normalization to map 132 features to a 16-D unit hypersphere.
2. **Domain-Adversarial Neural Network (DANN):** Uses a Gradient Reversal Layer (GRL) to align Madrid and Amsterdam feature distributions, eliminating city-specific reflectance bias.
3. **Metric Learning + Class Balancing:** Jointly optimizes era separation and handles severe historical class imbalance.
4. **Prototypical Few-Shot Transfer:** Adapts to Amsterdam using class prototypes for $n \in [5, 25, 50, 100, 200]$ samples per class over 10 randomized trials.
5. **Bayesian Uncertainty (MC-Dropout):** Quantifies predictive entropy and confidence across stochastic passes.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# ── Reproducibility & Device Configuration ────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Execution device: {DEVICE}")

### 4.2 Load Preprocessed Data
Loads the 132-dimensional feature matrices produced in `UnitModel_3_Preprocessing`.

In [ ]:
search_pkl = [
    'preprocessed_data.pkl',
    'data/preprocessed/preprocessed_data.pkl',
    '../data/preprocessed/preprocessed_data.pkl'
]
pkl_file = None
for p in search_pkl:
    if os.path.exists(p):
        pkl_file = p
        break

if pkl_file is None:
    raise FileNotFoundError(
        "preprocessed_data.pkl not found! Please run UnitModel_3_Preprocessing.ipynb first to generate features."
    )

print(f"Loading preprocessed features from: {pkl_file}")
with open(pkl_file, 'rb') as f:
    data = pickle.load(f)

X_madrid    = data['X_madrid']
y_madrid    = data['y_madrid']
X_amsterdam = data['X_amsterdam']
y_amsterdam = data['y_amsterdam']
feature_names = data.get('feature_names', [f'feat_{i}' for i in range(X_madrid.shape[1])])

print(f"Madrid Data: {X_madrid.shape}, Amsterdam Data: {X_amsterdam.shape}")
print(f"Madrid Class Counts: {np.bincount(y_madrid)[1:]}")
print(f"Amsterdam Class Counts: {np.bincount(y_amsterdam)[1:]}")

### 4.3 Model Architecture: `UnitMetricNet`
Deep embedding network with **Gradient Reversal Layer (GRL)** for domain-invariance and **L2 Hypersphere Normalization** for metric learning.

In [ ]:
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class DomainDiscriminator(nn.Module):
    def __init__(self, in_features=16, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

class UnitMetricNet(nn.Module):
    """
    Transferable UnitModel: City-Agnostic Deep Metric & Domain Adaptation Network.
    Maps engineered spectral features to an L2-normalized 16D unit hypersphere.
    """
    def __init__(self, input_dim=132, embedding_dim=16, num_classes=4, dropout=0.3):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(32, embedding_dim)
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)
        self.domain_discriminator = DomainDiscriminator(in_features=embedding_dim)

    def forward(self, x, alpha=1.0):
        raw_embed = self.backbone(x)
        # L2-normalize onto unit hypersphere
        norm_embed = F.normalize(raw_embed, p=2, dim=-1)
        
        # Age classification logits
        logits = self.classifier(norm_embed)
        
        # Domain prediction with Gradient Reversal
        rev_embed = GradientReversal.apply(norm_embed, alpha)
        domain_pred = self.domain_discriminator(rev_embed)
        
        return logits, norm_embed, domain_pred

    def get_embedding(self, x):
        raw_embed = self.backbone(x)
        return F.normalize(raw_embed, p=2, dim=-1)

print('UnitMetricNet architecture compiled successfully.')

### 4.4 Training Stage 1: Madrid Base Training with Domain Invariance
We train `UnitMetricNet` on Madrid labeled data while using Amsterdam unlabeled features to align the latent embedding space across cities.
We use **Class-Weighted Cross-Entropy** + **Supervised Metric Regularization** + **Domain Adversarial Loss**.

In [ ]:
# Hyperparameters
EPOCHS = 25
BATCH_SIZE = 256
LR = 0.001
WEIGHT_DECAY = 1e-4
LAMBDA_DOMAIN = 0.1
LAMBDA_METRIC = 0.2

# Class weights for Madrid imbalance
class_counts = np.bincount(y_madrid)[1:]
class_weights = torch.tensor(len(y_madrid) / (4.0 * class_counts), dtype=torch.float32).to(DEVICE)
ce_loss_fn = nn.CrossEntropyLoss(weight=class_weights)
bce_loss_fn = nn.BCELoss()

def supervised_metric_loss(embeddings, targets, margin=0.3):
    """
    Metric loss on unit hypersphere: encourages same-class pairs to have higher cosine similarity
    than different-class pairs by at least `margin`.
    """
    # Cosine similarity matrix
    sim_mat = torch.matmul(embeddings, embeddings.T)
    target_mat = targets.unsqueeze(1) == targets.unsqueeze(0)
    
    pos_mask = target_mat.fill_diagonal_(False)
    neg_mask = ~target_mat
    
    if pos_mask.sum() == 0 or neg_mask.sum() == 0:
        return torch.tensor(0.0, device=embeddings.device)
    
    # Sample hard positives and hard negatives
    pos_sim = sim_mat[pos_mask]
    neg_sim = sim_mat[neg_mask]
    
    # Margin loss: neg_sim - pos_sim + margin > 0
    loss = F.relu(neg_sim.unsqueeze(1) - pos_sim.unsqueeze(0) + margin).mean()
    return loss

# Prepare Datasets
X_mad_t = torch.tensor(X_madrid, dtype=torch.float32)
y_mad_t = torch.tensor(y_madrid - 1, dtype=torch.long)  # 0-indexed
X_ams_t = torch.tensor(X_amsterdam, dtype=torch.float32)

# 80/20 train/validation split on Madrid for internal validation
n_val = int(0.2 * len(X_mad_t))
indices = torch.randperm(len(X_mad_t))
train_idx, val_idx = indices[n_val:], indices[:n_val]

mad_train_loader = DataLoader(TensorDataset(X_mad_t[train_idx], y_mad_t[train_idx]), batch_size=BATCH_SIZE, shuffle=True)
mad_val_loader   = DataLoader(TensorDataset(X_mad_t[val_idx], y_mad_t[val_idx]), batch_size=BATCH_SIZE, shuffle=False)
ams_unlabeled_loader = DataLoader(TensorDataset(X_ams_t), batch_size=BATCH_SIZE, shuffle=True)

print(f"Training set: {len(train_idx):,} | Validation set: {len(val_idx):,}")

In [ ]:
model = UnitMetricNet(input_dim=X_madrid.shape[1], embedding_dim=16, num_classes=4, dropout=0.3).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f"Training UnitMetricNet on Madrid (with Amsterdam domain alignment) for {EPOCHS} epochs...")
ams_iter = iter(ams_unlabeled_loader)

for epoch in range(EPOCHS):
    model.train()
    p = float(epoch) / EPOCHS
    alpha = 2.0 / (1.0 + np.exp(-10 * p)) - 1.0  # Dynamic GRL schedule
    
    total_loss, total_ce, total_dom, total_met = 0.0, 0.0, 0.0, 0.0
    for batch_x_mad, batch_y_mad in mad_train_loader:
        batch_x_mad = batch_x_mad.to(DEVICE)
        batch_y_mad = batch_y_mad.to(DEVICE)
        
        # Get unlabeled Amsterdam batch
        try:
            batch_x_ams = next(ams_iter)[0].to(DEVICE)
        except StopIteration:
            ams_iter = iter(ams_unlabeled_loader)
            batch_x_ams = next(ams_iter)[0].to(DEVICE)
        
        # Madrid forward pass
        logits_mad, embed_mad, dom_pred_mad = model(batch_x_mad, alpha=alpha)
        ce_loss = ce_loss_fn(logits_mad, batch_y_mad)
        met_loss = supervised_metric_loss(embed_mad, batch_y_mad)
        
        # Amsterdam forward pass (domain only)
        _, _, dom_pred_ams = model(batch_x_ams, alpha=alpha)
        
        # Domain loss: Madrid = 0, Amsterdam = 1
        dom_labels_mad = torch.zeros_like(dom_pred_mad)
        dom_labels_ams = torch.ones_like(dom_pred_ams)
        dom_loss = 0.5 * (bce_loss_fn(dom_pred_mad, dom_labels_mad) + bce_loss_fn(dom_pred_ams, dom_labels_ams))
        
        loss = ce_loss + (LAMBDA_METRIC * met_loss) + (LAMBDA_DOMAIN * dom_loss)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_ce   += ce_loss.item()
        total_dom  += dom_loss.item()
        total_met  += met_loss.item()
        
    scheduler.step()
    
    # Validation on held-out Madrid pixels
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for val_x, val_y in mad_val_loader:
            val_x = val_x.to(DEVICE)
            val_logits, _, _ = model(val_x)
            val_preds.extend(val_logits.argmax(dim=-1).cpu().numpy())
            val_targets.extend(val_y.numpy())
            
    val_f1 = f1_score(val_targets, val_preds, average='macro')
    n_b = len(mad_train_loader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:2d}/{EPOCHS}] | Loss: {total_loss/n_b:.4f} (CE: {total_ce/n_b:.4f}, Met: {total_met/n_b:.4f}, Dom: {total_dom/n_b:.4f}) | Madrid Val Macro F1: {val_f1:.4f}")

### 4.5 Madrid Final Validation Results
Classification report and confusion matrix on Madrid.

In [ ]:
print("=== Madrid Internal Validation Performance ===")
print(f"Macro F1: {val_f1:.4f}")
print(classification_report(val_targets, val_preds, target_names=['Class 1', 'Class 2', 'Class 3', 'Class 4'], digits=4))

### 4.6 Zero-Shot Transfer to Amsterdam (Baseline Check)
Evaluating Madrid-trained model directly on Amsterdam without any Amsterdam labels.

In [ ]:
model.eval()
with torch.no_grad():
    ams_logits, _, _ = model(X_ams_t.to(DEVICE))
    ams_zero_pred = ams_logits.argmax(dim=-1).cpu().numpy() + 1

f1_zero = f1_score(y_amsterdam, ams_zero_pred, average='macro')
print(f"Zero-Shot Transfer to Amsterdam (No Amsterdam Labels): Macro F1 = {f1_zero:.4f}")

### 4.7 Stage 2: Prototypical Few-Shot Transfer to Amsterdam
We evaluate few-shot adaptation for `SHOTS_PER_CLASS = [5, 25, 50, 100, 200]` across 10 randomized trials.

**The Prototypical Mechanism:**
1. Given support set $S_k$ of size $n$ for each class $k \in \{1, 2, 3, 4\}$:
2. Compute unit-normalized prototypes in 16D embedding space: $c_k = \text{normalize}(\frac{1}{n} \sum_{x \in S_k} f(x))$
3. Query pixels are classified by minimum Euclidean distance (or maximum cosine similarity) to the prototypes.

In [ ]:
def extract_all_embeddings(model, tensor_x, batch_size=512):
    model.eval()
    embeds = []
    loader = DataLoader(TensorDataset(tensor_x), batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for bx in loader:
            e = model.get_embedding(bx[0].to(DEVICE))
            embeds.append(e.cpu().numpy())
    return np.vstack(embeds)

print("Extracting Madrid and Amsterdam embeddings in unit hypersphere space...")
E_madrid    = extract_all_embeddings(model, X_mad_t)
E_amsterdam = extract_all_embeddings(model, X_ams_t)
print(f"Embeddings ready: Madrid {E_madrid.shape}, Amsterdam {E_amsterdam.shape}")

In [ ]:
def prototypical_few_shot_eval(embeddings, labels, shots_list, n_trials=10, seed=42):
    rng = np.random.default_rng(seed)
    classes = np.unique(labels)
    results = {n: [] for n in shots_list}
    
    for n_shots in shots_list:
        for trial in range(n_trials):
            support_idx = []
            for c in classes:
                c_idx = np.where(labels == c)[0]
                chosen = rng.choice(c_idx, size=min(n_shots, len(c_idx)), replace=False)
                support_idx.extend(chosen)
            
            support_idx = np.array(support_idx)
            query_mask = np.ones(len(labels), dtype=bool)
            query_mask[support_idx] = False
            
            # Compute normalized class prototypes
            prototypes = np.stack([
                embeddings[support_idx[labels[support_idx] == c]].mean(axis=0) for c in classes
            ])
            # Normalize prototypes to unit hypersphere
            prototypes = prototypes / np.linalg.norm(prototypes, axis=1, keepdims=True)
            
            # Query set classification
            query_embeds = embeddings[query_mask]
            query_targets = labels[query_mask]
            
            # Cosine similarity (dot product on normalized vectors)
            sims = np.dot(query_embeds, prototypes.T)
            preds = classes[sims.argmax(axis=1)]
            
            score = f1_score(query_targets, preds, average='macro')
            results[n_shots].append(score)
            
    return results

SHOTS_PER_CLASS = [5, 25, 50, 100, 200]
N_TRIALS = 10

print(f"Evaluating Few-Shot Transfer across {SHOTS_PER_CLASS} samples/class ({N_TRIALS} trials each)...")
proto_results = prototypical_few_shot_eval(E_amsterdam, y_amsterdam, SHOTS_PER_CLASS, n_trials=N_TRIALS, seed=SEED)

print("\n=== PROTOTYPICAL FEW-SHOT TRANSFER RESULTS (MACRO F1) ===")
print("-----------------------------------------------------------")
for n in SHOTS_PER_CLASS:
    mean_f1 = np.mean(proto_results[n])
    std_f1  = np.std(proto_results[n])
    print(f"{n:3d} samples/class: Macro F1 = {mean_f1:.4f} +/- {std_f1:.4f}")
print("-----------------------------------------------------------")

### 4.8 Bayesian Uncertainty Quantification (Monte Carlo Dropout)
We enable dropout during inference over $M=20$ stochastic forward passes to compute predictive entropy $\mathcal{H}(p) = -\sum p_c \log p_c$.

In [ ]:
def enable_mc_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

def mc_dropout_predict(model, tensor_x, n_passes=20, batch_size=512):
    enable_mc_dropout(model)
    loader = DataLoader(TensorDataset(tensor_x), batch_size=batch_size, shuffle=False)
    all_probs = []
    with torch.no_grad():
        for _ in range(n_passes):
            pass_probs = []
            for bx in loader:
                logits, _, _ = model(bx[0].to(DEVICE))
                probs = F.softmax(logits, dim=-1).cpu().numpy()
                pass_probs.append(probs)
            all_probs.append(np.vstack(pass_probs))
    
    all_probs = np.stack(all_probs, axis=0) # [Passes, N, Classes]
    mean_probs = all_probs.mean(axis=0)
    # Predictive Entropy
    eps = 1e-8
    entropy = -np.sum(mean_probs * np.log(mean_probs + eps), axis=1)
    return mean_probs, entropy

print("Estimating Bayesian Uncertainty via MC-Dropout (20 passes)... ")
mean_probs, entropy = mc_dropout_predict(model, X_ams_t[:1000], n_passes=20)
print(f"Mean Predictive Entropy: {entropy.mean():.4f} (min={entropy.min():.4f}, max={entropy.max():.4f})")

### 4.9 Hackathon Submission Deliverable: Summary Table & Error-Bar Plot
Generates the official Macro F1 summary table and the required **Macro F1 vs. $\log_2(\text{sample size})$** plot for presentation slides.

In [ ]:
means = [np.mean(proto_results[n]) for n in SHOTS_PER_CLASS]
stds  = [np.std(proto_results[n])  for n in SHOTS_PER_CLASS]

# Summary Table in Markdown format
table_df = pd.DataFrame({
    'Samples per Class': SHOTS_PER_CLASS,
    'Total Target Labels': [n * 4 for n in SHOTS_PER_CLASS],
    'Macro F1 (Mean)': [f"{m:.4f}" for m in means],
    'Macro F1 (Std)': [f"{s:.4f}" for s in stds]
})
print("=== SUBMISSION DELIVERABLE: AMSTERDAM FEW-SHOT RESULTS ===")
print(table_df.to_string(index=False))

# Presentation Plot
plt.figure(figsize=(9, 5))
plt.errorbar(SHOTS_PER_CLASS, means, yerr=stds, fmt='o-', color='#1f77b4', linewidth=2.5, markersize=8, capsize=5, label='UnitMetricNet Prototypical Transfer')
plt.axhline(f1_zero, color='gray', linestyle='--', label=f'Zero-Shot Baseline (F1={f1_zero:.4f})')

plt.xscale('log', base=2)
plt.xticks(SHOTS_PER_CLASS, [str(n) for n in SHOTS_PER_CLASS])
plt.xlabel('Labeled Samples per Class (log2 scale)', fontsize=11, fontweight='bold')
plt.ylabel('Amsterdam Macro F1 Score', fontsize=11, fontweight='bold')
plt.title('Few-Shot Adaptation Performance vs. Local Annotation Volume', fontsize=13, fontweight='bold')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig('few_shot_transfer_f1_curve.png', dpi=300)
plt.show()
print('Saved publication-ready plot to few_shot_transfer_f1_curve.png.')